# Gulf FEI — Fuzzy Cognitive Map (FCM) Walkthrough

**Audience:** Gulf FEI research team
**Goal:** end-to-end reference for how the FCM subsystem turns research text into a signed causal map, and how to extend or audit it.

This notebook is generated from the live source tree by `FCM/docs/_build_notebook.py`. Every code cell labelled *Source* below is a verbatim copy of the production module — what you read here is what runs in the web app.

## Contents
1. What is a Fuzzy Cognitive Map?
2. System architecture
3. Data contract — `schemas.py`
4. Causal extraction — `extractor.py`
5. Graph assembly — `fcm_graph.py`
6. High-level orchestrator — `src/FCM.py`
7. Thematic categorisation — `categorizer.py`
8. Scenario simulation — `simulator.py`
9. End-to-end demo (runnable)
10. Quality controls and validation


## 1. What is a Fuzzy Cognitive Map?

A Fuzzy Cognitive Map (FCM, Kosko 1986) is a **signed, weighted, directed graph** whose nodes are domain *concepts* and whose edges encode *causal influence*:

- **Sign** (+/-) — does the source reinforce or oppose the target?
- **Weight** in `[-1, +1]` — how strong is the influence?
- **Direction** — which way does the causal arrow point?

Once a map is built, you can run *what-if* scenarios by clamping a subset of concepts (the **drivers**) and propagating activation through the matrix until the system reaches equilibrium. This makes FCMs a lightweight bridge between qualitative expert knowledge and quantitative simulation — exactly what NOAA's Ecosystem-Based Fisheries Management (EBFM) framework calls for in the Gulf of Mexico.

## 2. System architecture

```
 User question / uploaded file
        |
        v
  +--------------------+
  | RAG retrieval +    |   (src/rag.py, src/vector_loader.py)
  | LLM answer         |
  +--------------------+
        | text bundle (Q + A + retrieved context)
        v
  +--------------------+
  | CausalExtractor    |   (FCM/extractor.py)
  |  - regex patterns  |
  |  - co-occurrence   |
  |  - transitive 2hop |
  +--------------------+
        | concepts + signed edges
        v
  +--------------------+
  | FCMGraphBuilder    |   (FCM/fcm_graph.py)
  |  - adjacency mat.  |
  |  - exports         |
  +--------------------+
        | FCMMap
        v
  +--------------------+
  | FCMGenerator       |   (src/FCM.py)
  |  - networkx graph  |
  |  - publication plot|
  |  - thematic summary|
  +--------------------+
        | adjacency + plots
        v
  +--------------------+
  | FCMSimulator       |   (FCM/simulator.py)
  |  - Kosko propagate |
  |  - sigmoid/tanh    |
  +--------------------+
```


## 3. Data contract — `FCM/schemas.py`

Pydantic models that fix the shape of every object passed between stages. The two that matter most are `Edge` (a single causal claim with its evidence) and `FCMMap` (the assembled graph).


> **Source — `FCM/schemas.py`**

In [ ]:
from __future__ import annotations

from typing import Dict, List, Optional
from pydantic import BaseModel, Field


class DocumentIn(BaseModel):
    doc_id: str
    text: str
    metadata: Dict[str, str] = Field(default_factory=dict)


class IngestRequest(BaseModel):
    documents: List[DocumentIn]


class QueryRequest(BaseModel):
    query: str
    top_k: int = Field(default=5, ge=1, le=50)
    scenario_activation: Dict[str, float] = Field(default_factory=dict)
    run_simulation: bool = True
    max_relations: int = Field(default=10, ge=1, le=100)
    min_confidence: float = Field(default=0.0, ge=0.0, le=1.0)
    min_abs_weight: float = Field(default=0.0, ge=0.0, le=1.0)
    manual_relations: List['ManualRelationIn'] = Field(default_factory=list)


class RetrievedChunk(BaseModel):
    doc_id: str
    score: float
    text: str
    metadata: Dict[str, str] = Field(default_factory=dict)


class Edge(BaseModel):
    source: str
    target: str
    weight: float
    polarity: str
    confidence: float = 0.7
    evidence: str
    evidence_doc_id: Optional[str] = None
    edge_type: str = 'pattern'   # 'pattern' | 'cooccurrence' | 'transitive'
    hops: int = 1                 # 1 = direct, 2+ = transitive chain length


class ManualRelationIn(BaseModel):
    source: str
    target: str
    weight: float = Field(ge=-1.0, le=1.0)
    confidence: float = Field(default=0.95, ge=0.0, le=1.0)
    evidence: str = 'manual relation'


class FCMMap(BaseModel):
    concepts: List[str]
    edges: List[Edge]
    adjacency_matrix: List[List[float]]
    query: str


class AskResponse(BaseModel):
    query: str
    retrieved_chunks: List[RetrievedChunk]
    fcm_map: FCMMap
    simulation: Dict[str, float] = Field(default_factory=dict)
    answer: str
    artifacts: Dict[str, str] = Field(default_factory=dict)


## 4. Causal extraction — `FCM/extractor.py`

This is the heart of the system. Three complementary strategies feed the graph:

1. **Pattern matching** — a registry of compiled regexes (`CAUSAL_PATTERNS`) detects explicit causal verbs (*increases, reduces, leads to, due to, …*) and assigns a base weight and polarity to each match.
2. **Co-occurrence fallback** — when explicit patterns are sparse, any two domain keywords appearing in the same 3-sentence rolling window are connected. The window's dominant sentiment (`_NEG_MARKERS` vs `_POS_MARKERS`) sets the sign so co-occurrence edges are not blindly positive.
3. **Transitive closure** — a damped 2-hop inference fires `A → B → C ⇒ A → C` so reviewers can see second-order pathways.

Concept names pass through two strict filters:

- `normalize_concept` — strips articles, useless modifiers from either end, dedupes consecutive duplicates, title-cases while preserving acronyms.
- `_is_clean_concept` — rejects clause fragments, interrogatives anywhere in the span, all-modifier phrases, vague single-word nouns, and >50%-stopword spans.


> **Source — `FCM/extractor.py`**

In [ ]:
from __future__ import annotations

import re
from collections import defaultdict
from difflib import SequenceMatcher
from typing import Dict, Iterable, List, Tuple

from .schemas import Edge

# Domain acronyms that must never be title-cased or lemmatized (e.g. "Cpue" or "Cpu")
_ACRONYMS = {
    'CPUE', 'MSY', 'EBM', 'EBFM', 'IUU', 'TAC', 'NOAA', 'NMFS', 'GOM',
    'FEP', 'MPA', 'GDP', 'SST', 'DO', 'ENSO', 'DOC', 'EPA', 'USCG',
    'ITQ', 'ACL', 'ABC', 'OFL', 'MRIP', 'SEAMAP', 'HAB', 'HABs',
}

# ---------------------------------------------------------------------------
# Pattern templates
#
#   _FIRST  – non-greedy: matches the concept BEFORE the causal verb.
#             Non-greedy ensures it does not consume the verb itself.
#
#   _LAST   – greedy up to 4 words, stops before common conjunctions/
#             relative pronouns so the effect phrase doesn't bleed into the
#             next clause ("fish stock abundance" not "fish stock abundance
#             and threatens marine biodiversity").
#
# Both templates accept {g} as the regex group name; all other braces that
# would confuse str.format() are doubled.
# ---------------------------------------------------------------------------

_STOP_CONJ = (
    # Conjunctions & relative pronouns
    r'and|or|but|which|that|because|since|while|however|'
    r'although|if|when|where|as|so|yet|nor|'
    # Prepositions that end a noun phrase
    r'of|in|on|at|by|for|with|from|into|onto|upon|via|'
    # Causal markers (stop before "due to", "because of", etc.)
    r'due|owing|through|throughout'
)

# Single word: letters, digits, hyphens, slashes (NO spaces — keeps word boundaries clean)
_W = r'[A-Za-z][A-Za-z0-9\-/]*'

# Cause / driver: 1-4 words, non-greedy (stops as early as possible before the verb)
_FIRST = (
    r'(?P<{g}>'
    + _W +
    r'(?:\s+' + _W + r'){{0,3}}?'
    r')'
)

# Effect / outcome: 1-4 words, greedy, stops before conjunctions / punctuation
_LAST = (
    r'(?P<{g}>'
    + _W +
    r'(?:\s+(?!\b(?:' + _STOP_CONJ + r')\b)' + _W + r'){{0,3}}'
    r')'
)

# Convenience aliases (active patterns: A verb B)
_A = _FIRST
_B = _LAST

# If a concept's FIRST word is one of these, it is a captured predicate, not a noun phrase.
# Words that are never the head of a proper noun phrase in this domain.
_VERB_FIRST_WORDS = {
    # Base forms
    'increases', 'increase', 'reduces', 'reduce', 'causes', 'cause',
    'drives', 'drive', 'leads', 'lead', 'affects', 'affect',
    'threatens', 'threaten', 'promotes', 'promote', 'damages', 'damage',
    'prevents', 'prevent', 'limits', 'limit', 'enhances', 'enhance',
    'inhibits', 'inhibit', 'produces', 'produce', 'creates', 'create',
    'generates', 'generate', 'contributes', 'contribute',
    'triggers', 'trigger', 'impacts', 'impact',
    # Past tense / gerund forms that sneak through as nouns
    'caused', 'causing', 'increased', 'increasing', 'decreased', 'decreasing',
    'reduced', 'reducing', 'driven', 'driving', 'leading', 'affecting',
    'threatened', 'threatening', 'promoted', 'promoting', 'damaged', 'damaging',
    'prevented', 'preventing', 'limited', 'limiting', 'enhanced', 'enhancing',
    'inhibited', 'inhibiting', 'produced', 'producing', 'created', 'creating',
    'generated', 'generating', 'contributed', 'contributing',
    'triggered', 'triggering', 'impacted', 'impacting',
    'mitigates', 'mitigate', 'mitigated', 'mitigating',
    'results', 'result', 'resulted', 'resulting',
    # Question/interrogative starts — never valid FCM concepts
    'how', 'what', 'why', 'when', 'where', 'does', 'did', 'do',
    'is', 'are', 'was', 'were', 'can', 'could', 'should', 'would',
    # Pronouns / determiners — never a concept head
    'it', 'its', 'this', 'that', 'these', 'those', 'they', 'their',
    'he', 'she', 'we', 'you', 'i', 'such', 'both', 'each', 'all',
    # Prepositions / conjunctions that regex can grab as "first word"
    'of', 'in', 'on', 'at', 'by', 'for', 'with', 'from', 'into', 'onto',
    'via', 'due', 'as', 'or', 'and', 'but', 'not', 'also', 'more', 'most',
}

# Words that, when found ANYWHERE in the concept, signal a predicate fragment
_CLAUSE_VERBS = {
    'associated', 'linked', 'correlated', 'determines', 'mitigates', 'alleviates',
    'damages', 'threatens', 'reduces', 'promotes', 'prevents', 'inhibits',
}

# Trailing words that make a concept fragment — strip these from the end
_TRAILING_STRIP = re.compile(
    r'\s+\b(?:of|in|on|at|by|for|with|from|into|to|the|a|an|'
    r'and|or|but|which|that|this|these|those|its|their)\b$',
    re.I,
)

# Single-word concepts that are too vague to carry FCM meaning
_VAGUE_SINGLES = {
    'level', 'levels', 'rate', 'rates', 'amount', 'amounts', 'number', 'numbers',
    'factor', 'factors', 'effect', 'effects', 'impact', 'impacts', 'change', 'changes',
    'increase', 'decrease', 'result', 'results', 'role', 'part', 'aspect', 'issue',
    'area', 'areas', 'type', 'types', 'form', 'forms', 'way', 'ways', 'case', 'cases',
    'process', 'processes', 'system', 'systems', 'term', 'terms', 'use', 'used',
    'value', 'values', 'range', 'basis', 'data', 'point', 'points', 'need', 'needs',
    'example', 'examples', 'situation', 'situations', 'condition', 'conditions',
    'period', 'time', 'year', 'years', 'month', 'months', 'day', 'days',
    # Generic placeholder nouns that carry no domain content
    'thing', 'things', 'stuff', 'matter', 'matters', 'kind', 'kinds',
    'sort', 'sorts', 'lot', 'lots', 'set', 'sets', 'group', 'groups',
    'side', 'sides', 'end', 'ends', 'start', 'starts', 'beginning',
    'place', 'places', 'person', 'people', 'someone', 'anyone', 'everyone',
    'item', 'items', 'object', 'objects', 'section', 'sections',
    'chapter', 'chapters', 'page', 'pages', 'version', 'versions',
    'name', 'names', 'word', 'words', 'idea', 'ideas', 'concept', 'concepts',
    'topic', 'topics', 'subject', 'subjects', 'detail', 'details',
    'information', 'info', 'content', 'overview', 'summary', 'background',
    'yes', 'no', 'maybe', 'ok', 'okay',
    # Conversational/meta filler that occasionally slips out of the regex
    'note', 'notes', 'question', 'questions', 'answer', 'answers',
    'thanks', 'please', 'sure', 'fine', 'good', 'bad',
}

# ---------------------------------------------------------------------------
# Causal pattern registry
# Each entry: (compiled_regex, base_weight, polarity_label)
# ---------------------------------------------------------------------------

def _pat(pattern_str: str) -> re.Pattern:
    return re.compile(pattern_str, re.I)


CAUSAL_PATTERNS: List[Tuple[re.Pattern, float, str]] = [

    # ── Strong positive: X increases / boosts / enhances Y ─────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:significantly\s+|substantially\s+|greatly\s+)?'
            r'(?:increases?|raises?|boosts?|improves?|strengthens?|enhances?|promotes?|'
            r'facilitates?|stimulates?|expands?|accelerates?|drives?|elevates?|amplifies?|'
            r'maximizes?|supports?)\s+' +
            _B.format(g='b')
        ), 0.75, 'positive',
    ),

    # ── Strong negative: X reduces / threatens / damages Y ──────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:significantly\s+|substantially\s+|greatly\s+)?'
            r'(?:decreases?|reduces?|weakens?|lowers?|depletes?|degrades?|'
            r'damages?|harms?|threatens?|destroys?|inhibits?|suppresses?|'
            r'diminishes?|impairs?|undermines?|hurts?|minimizes?|'
            r'disrupts?|disturbs?|erodes?|worsens?|aggravates?|exacerbates?|'
            r'endangers?|jeopardizes?|disrupted?|kills?|eliminates?|'
            r'conflicts?\s+with|interferes?\s+with|competes?\s+with)\s+' +
            _B.format(g='b')
        ), -0.75, 'negative',
    ),

    # ── "negatively / adversely affects|impacts|influences" ─────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:negatively|adversely|detrimentally|harmfully)\s+'
            r'(?:affects?|impacts?|influences?|alters?|changes?|shapes?|'
            r'modifies?|correlates?|relates?)\s+(?:with\s+|to\s+)?' +
            _B.format(g='b')
        ), -0.70, 'negative',
    ),

    # ── "X causes loss / decline / decrease in Y" ──────────────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:causes?|drives?|leads?\s+to|results?\s+in|produces?|triggers?)\s+'
            r'(?:a\s+|the\s+)?(?:loss|decline|decrease|drop|fall|reduction|'
            r'depletion|degradation|deterioration|collapse|extinction|'
            r'destruction|disruption)\s+(?:of|in)\s+' +
            _B.format(g='b')
        ), -0.70, 'negative',
    ),

    # ── Direct causation: X leads to / causes / triggers Y ─────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:leads?\s+to|results?\s+in|causes?|triggers?|induces?|generates?|'
            r'produces?|creates?|contributes?\s+to|gives?\s+rise\s+to|brings?\s+about)\s+' +
            _B.format(g='b')
        ), 0.65, 'positive',
    ),

    # ── Negative control: X prevents / limits Y ─────────────────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:prevents?|limits?|constrains?|restricts?|blocks?|'
            r'mitigates?|alleviates?|controls?|curbs?|halts?|stops?)\s+' +
            _B.format(g='b')
        ), -0.65, 'negative',
    ),

    # ── Affect / influence (direction inferred as positive, moderate weight) ─
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:affects?|impacts?|influences?|alters?|changes?|shapes?|'
            r'modifies?|determines?|governs?|drives?)\s+' +
            _B.format(g='b')
        ), 0.55, 'positive',
    ),

    # ── Association / linkage ───────────────────────────────────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:is\s+|are\s+)?(?:positively\s+)?'
            r'(?:associated\s+with|linked\s+to|correlated\s+with|'
            r'connected\s+to|related\s+to)\s+' +
            _B.format(g='b')
        ), 0.50, 'positive',
    ),

    # ── Negative association ────────────────────────────────────────────────
    (
        _pat(
            _A.format(g='a') +
            r'\s+(?:is\s+|are\s+)?negatively\s+'
            r'(?:associated\s+with|linked\s+to|correlated\s+with)\s+' +
            _B.format(g='b')
        ), -0.50, 'negative',
    ),

    # ── Passive positive: "B is increased/driven by A"  →  A causes B ───────
    (
        _pat(
            _FIRST.format(g='b') +
            r'\s+(?:is|are|was|were)\s+'
            r'(?:increased|driven|caused|triggered|induced|enhanced|'
            r'boosted|promoted|generated|elevated)\s+by\s+' +
            _LAST.format(g='a')
        ), 0.65, 'positive',
    ),

    # ── Passive negative: "B is reduced/threatened by A"  →  A harms B ─────
    (
        _pat(
            _FIRST.format(g='b') +
            r'\s+(?:is|are|was|were)\s+'
            r'(?:reduced|depleted|degraded|damaged|threatened|harmed|'
            r'decreased|limited|impaired|suppressed|undermined)\s+by\s+' +
            _LAST.format(g='a')
        ), -0.65, 'negative',
    ),

    # ── "B due to / because of / as a result of A"  →  A causes B ──────────
    (
        _pat(
            _FIRST.format(g='b') +
            r'\s+(?:due\s+to|because\s+of|as\s+a\s+result\s+of|'
            r'owing\s+to|in\s+response\s+to)\s+' +
            _LAST.format(g='a')
        ), 0.60, 'positive',
    ),
]

# ---------------------------------------------------------------------------
# Stop-words & domain keywords
# ---------------------------------------------------------------------------

STOPWORDS = {
    'the', 'a', 'an', 'this', 'that', 'these', 'those', 'it', 'they',
    'he', 'she', 'we', 'you', 'i', 'and', 'or', 'but', 'if', 'for',
    'to', 'of', 'in', 'on', 'with', 'by', 'from', 'at', 'as', 'its',
    'their', 'our', 'has', 'have', 'had', 'be', 'been', 'being',
    'is', 'are', 'was', 'were', 'can', 'could', 'may', 'might',
    'will', 'would', 'shall', 'should', 'such', 'which', 'when',
    # Interrogatives and auxiliaries — must never appear inside a concept name
    'how', 'what', 'why', 'where', 'does', 'did', 'do', 'not',
    'also', 'both', 'each', 'more', 'most', 'other', 'some',
    'than', 'then', 'there', 'so', 'very', 'just', 'about',
    # Empty intensifiers / qualifiers that add no FCM meaning
    'much', 'many', 'few', 'little', 'lots', 'plenty',
    'really', 'quite', 'rather', 'simply', 'only', 'even', 'still', 'yet',
    'almost', 'nearly', 'maybe', 'perhaps', 'probably', 'possibly',
    'always', 'never', 'often', 'sometimes', 'usually',
    'here', 'now', 'today', 'tomorrow', 'yesterday',
}

# Words that are pure modifiers/qualifiers — strip from the start or end of a
# concept regardless of position; never count as the meaningful head of a phrase.
_USELESS_MODIFIERS = {
    'very', 'much', 'more', 'most', 'less', 'least', 'many', 'few',
    'some', 'any', 'all', 'every', 'each', 'such', 'no',
    'just', 'only', 'even', 'still', 'yet', 'also',
    'really', 'quite', 'rather', 'simply', 'almost', 'nearly',
    'maybe', 'perhaps', 'probably', 'possibly',
    'always', 'never', 'often', 'sometimes', 'usually',
    'high', 'low', 'big', 'small', 'large', 'huge', 'tiny',  # bare adjectives w/o noun
    'good', 'bad', 'better', 'worse', 'best', 'worst',
    'new', 'old', 'recent',
}

# If ANY word inside a concept matches these, the span is a clause fragment,
# not a noun phrase — reject the whole concept.
_DISALLOWED_ANYWHERE = {
    # Interrogatives anywhere
    'how', 'what', 'why', 'when', 'where', 'who', 'whom', 'whose', 'which',
    # Auxiliaries / modals anywhere — signals a clause
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'am',
    'do', 'does', 'did', 'doing', 'done',
    'has', 'have', 'had', 'having',
    'can', 'could', 'shall', 'should', 'will', 'would', 'may', 'might', 'must',
    # Pronouns anywhere
    'it', 'its', 'they', 'them', 'their', 'theirs',
    'he', 'him', 'his', 'she', 'her', 'hers',
    'we', 'us', 'our', 'ours', 'you', 'your', 'yours', 'i', 'me', 'my', 'mine',
    'this', 'that', 'these', 'those',
    # Negation anywhere — hides causal sign
    'not', 'never', "n't",
}

# Sentence-level polarity cues: used by the co-occurrence fallback so weak
# edges inherit the negative tone of their sentence instead of being forced
# positive by default.
_NEG_MARKERS = re.compile(
    r'\b(?:decline|declin\w+|decrease|decreas\w+|reduc\w+|loss|losses|lost|'
    r'lower|lowered|damag\w+|threat\w+|destroy\w+|degrad\w+|deplet\w+|'
    r'disrupt\w+|harm\w+|kill\w+|collapse|overfish\w+|pollut\w+|'
    r'contaminat\w+|extinct\w+|endanger\w+|deterior\w+|impair\w+|'
    r'less|fewer|suppress\w+|inhibit\w+|prevent\w+|limit\w+|restrict\w+|'
    r'conflict\w*|interfere\w*|compete[ds]?|bycatch|hypoxi\w+|'
    r'negative\w*|adverse\w*|detriment\w*|worsen\w*|aggravat\w+|exacerbat\w+|'
    r'mortality|die-off|dying|extinction|crisis|crash|crashed)\b',
    re.IGNORECASE,
)

_POS_MARKERS = re.compile(
    r'\b(?:increase\w*|grow\w+|rise|rising|rose|boost\w*|improv\w+|enhanc\w+|'
    r'strengthen\w*|recover\w+|restore\w+|restoration|sustain\w+|benefit\w*|'
    r'support\w*|promot\w+|foster\w+|facilitat\w+|protect\w+|conserv\w+|'
    r'rebuild\w*|expand\w+|thriv\w+|flourish\w+|positive\w*)\b',
    re.IGNORECASE,
)


def _sentence_polarity(sentence: str) -> int:
    """Return -1 if the sentence leans negative, +1 otherwise."""
    neg = len(_NEG_MARKERS.findall(sentence))
    pos = len(_POS_MARKERS.findall(sentence))
    return -1 if neg > pos else 1


def _interleave_polarity(edges: List['Edge']) -> List['Edge']:
    """Re-rank so positive and negative edges alternate at the top.

    A credible signed FCM must not be dominated by one polarity in the
    first N shown edges. This preserves the overall ranking within each
    polarity but interleaves them 2:1 (positive-heavy) so negatives are
    visible without overwhelming the typical positive signal.
    """
    pos = [e for e in edges if e.weight >= 0]
    neg = [e for e in edges if e.weight < 0]
    if not neg or not pos:
        return edges
    out: List['Edge'] = []
    pi = ni = 0
    # 2 positive : 1 negative cadence — honest balance without flipping tone
    while pi < len(pos) or ni < len(neg):
        for _ in range(2):
            if pi < len(pos):
                out.append(pos[pi]); pi += 1
        if ni < len(neg):
            out.append(neg[ni]); ni += 1
    return out


_DOMAIN_KW = re.compile(
    r'\b(fish(?:ing|eries?|stock)?|bycatch|trawl(?:ing)?|harvest(?:ing)?|'
    r'spawn(?:ing)?|recruit(?:ment)?|biomass|abundance|mortality|overfishing|'
    r'ecosystem|habitat|coral|reef|mangrove|seagrass|tuna|shrimp|lobster|crab|'
    r'snapper|grouper|marine|ocean(?:ic)?|gulf|coastal|aquaculture|climate|'
    r'temperature|salinity|pollution|nutrient|oxygen|hypox\w+|management|'
    r'regulation|quota|effort|vessel|gear|discard|population|species|'
    r'biodiversity|conservation|sustainability|stock|assessment|CPUE|MSY|'
    r'EBM|EBFM|ecosystem.based|fishing\s+pressure|sea\s+level|water\s+quality)\b',
    re.I,
)


def _is_clean_concept(concept: str) -> bool:
    """Return False if the concept looks like a captured clause fragment."""
    if not concept:
        return False
    words = concept.lower().split()
    if len(words) > 4:
        return False
    # Must be at least 4 characters long (rejects "Up", "A", "On", etc.)
    if len(concept) < 4:
        return False
    # Reject if the first word is a causal verb, gerund, past tense, pronoun or preposition
    if words[0] in _VERB_FIRST_WORDS:
        return False
    # Reject if the last word is a dangling preposition or conjunction
    _TRAILING_PREPS = {
        'of', 'in', 'on', 'at', 'by', 'for', 'with', 'from', 'into',
        'to', 'the', 'a', 'an', 'and', 'or', 'but',
    }
    if words[-1] in _TRAILING_PREPS:
        return False
    # Reject single-word concepts that are pure past-participle adjectives
    # ("increased", "reducing") UNLESS the word is a domain noun-gerund
    # ("overfishing", "fishing", "spawning") — those are legitimate FCM concepts.
    if (len(words) == 1
            and words[0].endswith(('ed', 'ing', 'ise', 'ize'))
            and not _DOMAIN_KW.search(words[0])):
        return False
    # Reject single-word concepts that are too vague to mean anything alone
    if len(words) == 1 and words[0] in _VAGUE_SINGLES:
        return False
    # Reject if the concept is entirely made of stopwords
    non_stop = [w for w in words if w not in STOPWORDS]
    if not non_stop:
        return False
    # Reject pure-numeric concepts ("2", "100", "3.5")
    if re.fullmatch(r'[\d.,\-/]+', concept.replace(' ', '')):
        return False
    # Reject if any word is a clear passive/relational verb marker
    if set(words) & _CLAUSE_VERBS:
        return False
    # Reject if any word is an interrogative, auxiliary, pronoun, or negation
    # (these are clause markers, never noun-phrase content) — applies to ANY
    # position, not just the first word.
    word_set = set(words)
    if word_set & _DISALLOWED_ANYWHERE:
        return False
    # Reject if every word is a useless modifier (e.g. "Very High", "Much More")
    if all(w in _USELESS_MODIFIERS or w in STOPWORDS for w in words):
        return False
    # Reject if more than half the words are stopwords — concept is mostly filler
    stop_count = sum(1 for w in words if w in STOPWORDS)
    if len(words) >= 2 and stop_count * 2 > len(words):
        return False
    # Reject if any token is a sub-2-letter junk fragment (regex sometimes
    # captures stray "a", "I", initials, etc. that survived earlier filters)
    if any(len(w) < 2 for w in words):
        return False
    # Reject duplicate-word concepts ("Fish Fish", "Stock Stock")
    if len(words) >= 2 and len(set(words)) == 1:
        return False
    return True


def _smart_title(text: str) -> str:
    """Title-case text but preserve domain acronyms (CPUE, MSY, NOAA, ...)."""
    out: List[str] = []
    for w in text.split():
        up = w.upper()
        if up in _ACRONYMS:
            out.append(up)
        else:
            out.append(w.capitalize())
    return ' '.join(out)


def normalize_concept(text: str) -> str:
    cleaned = re.sub(r'\s+', ' ', text.strip(" .,:;\n\t'\"()[]"))
    # Strip leading articles
    cleaned = re.sub(r'^(?:the|a|an)\s+', '', cleaned, flags=re.I)
    # Strip trailing prepositions / conjunctions / articles iteratively
    prev = None
    while prev != cleaned:
        prev = cleaned
        cleaned = _TRAILING_STRIP.sub('', cleaned).strip()
    # Remove internal stopwords but keep domain multi-word phrases intact
    words = cleaned.split()
    # Only filter stopwords from single-word results; keep multi-word as-is
    # so that "Fish Stock" and "Water Quality" survive
    if len(words) == 1 and words[0].lower() in STOPWORDS:
        return ''
    # Strip leading stopwords / useless modifiers iteratively
    # ("very high fish stock" → "fish stock", "more of overfishing" → "overfishing")
    while len(words) >= 2 and (
        words[0].lower() in STOPWORDS
        or words[0].lower() in _USELESS_MODIFIERS
    ):
        words = words[1:]
    # Strip trailing useless modifiers / bare adjectives that were not caught
    # by _TRAILING_STRIP ("fish stock high" → "fish stock")
    while len(words) >= 2 and words[-1].lower() in _USELESS_MODIFIERS:
        words = words[:-1]
    # Collapse consecutive duplicate tokens ("fish fish stock" → "fish stock")
    deduped: List[str] = []
    for w in words:
        if not deduped or deduped[-1].lower() != w.lower():
            deduped.append(w)
    words = deduped
    cleaned = ' '.join(words)
    return _smart_title(cleaned) if len(cleaned) >= 4 else ''


# ---------------------------------------------------------------------------
# Lemmatization + fuzzy-merge for duplicate consolidation
# ---------------------------------------------------------------------------

def _lemma_token(word: str) -> str:
    """Crude stemmer — strips plural endings. Acronyms untouched."""
    if word.upper() in _ACRONYMS:
        return word.upper()
    w = word.lower()
    if len(w) > 4 and w.endswith('ies'):
        return w[:-3] + 'y'
    if len(w) > 4 and w.endswith('es') and not w.endswith(('ses', 'ches', 'shes', 'xes', 'zes')):
        return w[:-2]
    if len(w) > 3 and w.endswith('s') and not w.endswith('ss'):
        return w[:-1]
    return w


def _lemma_key(name: str) -> str:
    """Canonical key for duplicate detection: lemmatized + sorted tokens."""
    tokens = [_lemma_token(w) for w in name.split()]
    return ' '.join(sorted(tokens))


def fuzzy_merge_concepts(
    concepts: List[str],
    edges: List[Edge],
    similarity: float = 0.88,
) -> Tuple[List[str], List[Edge]]:
    """Consolidate near-duplicate concepts and merge their edges.

    Two concepts are collapsed when they share a lemma key OR their string
    similarity ≥ `similarity`. The canonical form is the longest name in the
    cluster (more informative), preserving acronyms. Edges between the same
    (source, target) post-merge are combined by confidence-weighted mean.
    """
    if len(concepts) < 2:
        return concepts, edges

    # Stage 1: bucket by lemma key
    lemma_buckets: Dict[str, List[str]] = defaultdict(list)
    for c in concepts:
        lemma_buckets[_lemma_key(c)].append(c)

    # Stage 2: fuzzy-match any remaining singletons against other lemma keys
    canon: Dict[str, str] = {}        # original name -> canonical name
    canonical_names: List[str] = []
    for names in lemma_buckets.values():
        head = max(names, key=len)    # prefer longer, more informative form
        canonical_names.append(head)
        for n in names:
            canon[n] = head

    # Collapse near-duplicate canonical names by string similarity
    merged_heads: Dict[str, str] = {}
    resolved: List[str] = []
    for name in canonical_names:
        chosen = None
        for existing in resolved:
            ratio = SequenceMatcher(None, name.lower(), existing.lower()).ratio()
            if ratio >= similarity:
                chosen = existing if len(existing) >= len(name) else name
                if chosen != existing:
                    merged_heads[existing] = chosen
                    resolved[resolved.index(existing)] = chosen
                else:
                    merged_heads[name] = chosen
                break
        if chosen is None:
            resolved.append(name)
            merged_heads[name] = name

    # Chase merged_heads chains to final canonical
    def _final(n: str) -> str:
        seen = set()
        while n in merged_heads and merged_heads[n] != n and n not in seen:
            seen.add(n)
            n = merged_heads[n]
        return n

    for orig, head in list(canon.items()):
        canon[orig] = _final(head)

    # Stage 3: re-map edges and merge duplicates
    grouped: Dict[Tuple[str, str], List[Edge]] = defaultdict(list)
    for e in edges:
        src = canon.get(e.source, e.source)
        tgt = canon.get(e.target, e.target)
        if src == tgt:
            continue
        grouped[(src, tgt)].append(e)

    new_edges: List[Edge] = []
    for (src, tgt), group in grouped.items():
        total_conf = sum(x.confidence for x in group)
        mean_w = sum(x.weight * x.confidence for x in group) / total_conf
        mean_c = total_conf / len(group)
        evidence = ' | '.join(dict.fromkeys(x.evidence for x in group))
        # Preserve the strongest edge_type label (pattern > cooccurrence > transitive)
        type_priority = {'pattern': 3, 'cooccurrence': 2, 'transitive': 1}
        best_type = max((x.edge_type for x in group), key=lambda t: type_priority.get(t, 0))
        min_hops = min(x.hops for x in group)
        new_edges.append(Edge(
            source=src,
            target=tgt,
            weight=round(mean_w, 3),
            polarity='positive' if mean_w >= 0 else 'negative',
            confidence=round(mean_c, 3),
            evidence=evidence,
            evidence_doc_id=group[0].evidence_doc_id,
            edge_type=best_type,
            hops=min_hops,
        ))

    new_edges = _dedupe_bidirectional(new_edges)
    new_edges.sort(key=lambda e: e.confidence * abs(e.weight), reverse=True)
    new_concepts = list(dict.fromkeys(canon[c] for c in concepts))
    return new_concepts, new_edges


def _dedupe_bidirectional(edges: List[Edge]) -> List[Edge]:
    """Collapse A→B and B→A when they carry the same polarity.

    Same-polarity reverse pairs encode the same causal meaning (especially
    from the undirected co-occurrence fallback), so we keep only the
    stronger direction (confidence × |weight|). Opposing polarity is left
    intact — that's a legitimate feedback loop in an FCM.
    """
    by_pair: Dict[Tuple[str, str], Edge] = {(e.source, e.target): e for e in edges}
    dropped: set = set()
    for (src, tgt), edge in list(by_pair.items()):
        rev_key = (tgt, src)
        if rev_key in dropped or (src, tgt) in dropped:
            continue
        rev = by_pair.get(rev_key)
        if rev is None:
            continue
        same_sign = (edge.weight >= 0) == (rev.weight >= 0)
        if not same_sign:
            continue  # genuine opposing feedback — keep both
        score_fwd = edge.confidence * abs(edge.weight)
        score_rev = rev.confidence * abs(rev.weight)
        loser = rev_key if score_fwd >= score_rev else (src, tgt)
        dropped.add(loser)
    return [e for e in edges if (e.source, e.target) not in dropped]


# ---------------------------------------------------------------------------
# Main extractor class
# ---------------------------------------------------------------------------

class CausalExtractor:
    def extract(self, chunks: Iterable[dict]) -> Tuple[List[str], List[Edge]]:
        concepts: Dict[str, None] = {}
        grouped: Dict[Tuple[str, str], List[Edge]] = defaultdict(list)
        all_text_parts: List[str] = []

        for chunk in chunks:
            text = chunk.get('text', '')
            all_text_parts.append(text)

            for sentence in re.split(r'(?<=[.!?])\s+', text):
                for pattern, base_weight, polarity in CAUSAL_PATTERNS:
                    for match in pattern.finditer(sentence):
                        source = normalize_concept(match.group('a'))
                        target = normalize_concept(match.group('b'))
                        if not source or not target or source == target:
                            continue
                        # Skip if concept looks like a captured clause fragment
                        if not _is_clean_concept(source) or not _is_clean_concept(target):
                            continue
                        concepts[source] = None
                        concepts[target] = None
                        edge = Edge(
                            source=source,
                            target=target,
                            weight=base_weight,
                            polarity=polarity,
                            confidence=min(0.95, 0.55 + float(chunk.get('score', 0.0))),
                            evidence=sentence.strip(),
                            evidence_doc_id=chunk.get('doc_id'),
                            edge_type='pattern',
                            hops=1,
                        )
                        grouped[(source, target)].append(edge)

        # Merge duplicate edges by confidence-weighted mean
        merged_edges: List[Edge] = []
        for (source, target), edges in grouped.items():
            total_conf = sum(e.confidence for e in edges)
            mean_weight = sum(e.weight * e.confidence for e in edges) / total_conf
            mean_conf = total_conf / len(edges)
            evidence = ' | '.join(dict.fromkeys(e.evidence for e in edges))
            merged_edges.append(
                Edge(
                    source=source,
                    target=target,
                    weight=round(mean_weight, 3),
                    polarity='positive' if mean_weight >= 0 else 'negative',
                    confidence=round(mean_conf, 3),
                    evidence=evidence,
                    evidence_doc_id=edges[0].evidence_doc_id,
                    edge_type='pattern',
                    hops=1,
                )
            )

        # Sort by (confidence × |weight|) descending — best pattern edges first
        merged_edges.sort(key=lambda e: e.confidence * abs(e.weight), reverse=True)

        # ── Always supplement with co-occurrence edges ───────────────────────
        # When pattern matches are few (or zero), co-occurrence fills the gap
        # so that higher relation-length requests return more results.
        supp_edges, supp_concepts = self._cooccurrence_fallback(all_text_parts, dict(concepts))

        if not merged_edges:
            # No explicit patterns at all — use co-occurrence only
            sc, se = fuzzy_merge_concepts(list(supp_concepts.keys()), supp_edges)
            return sc, _interleave_polarity(se)
        else:
            # Append co-occurrence edges that don't duplicate a pattern edge
            existing_pairs = {(e.source, e.target) for e in merged_edges}
            for e in supp_edges:
                if (e.source, e.target) not in existing_pairs:
                    merged_edges.append(e)
                    existing_pairs.add((e.source, e.target))
                    concepts[e.source] = None
                    concepts[e.target] = None
            # Re-sort: pattern-matched (high confidence) first, then interleave
            merged_edges.sort(key=lambda e: e.confidence * abs(e.weight), reverse=True)

        # Consolidate near-duplicate concepts (Plurals, acronyms, fuzzy matches)
        concept_list = list(concepts.keys())
        concept_list, merged_edges = fuzzy_merge_concepts(concept_list, merged_edges)

        # ── Transitive closure: A→B + B→C ⇒ A→C (damped, 2-hop inferred) ──
        # Densifies the graph and surfaces second-order causal pathways that
        # researchers can spot-check against the cited evidence.
        inferred = self._transitive_closure(merged_edges)
        if inferred:
            existing = {(e.source, e.target) for e in merged_edges}
            for e in inferred:
                if (e.source, e.target) not in existing:
                    merged_edges.append(e)
                    existing.add((e.source, e.target))

        merged_edges = _dedupe_bidirectional(merged_edges)
        merged_edges.sort(key=lambda e: e.confidence * abs(e.weight), reverse=True)
        merged_edges = _interleave_polarity(merged_edges)
        return concept_list, merged_edges

    # ------------------------------------------------------------------
    # Fallback: domain keyword co-occurrence (3-sentence rolling window,
    # all-pairs within the window). Wider context = denser, more
    # informative graphs; weights scale by window proximity.
    # ------------------------------------------------------------------
    def _cooccurrence_fallback(
        self,
        text_parts: List[str],
        concepts: Dict[str, None],
        window: int = 3,
    ) -> Tuple[List[Edge], Dict[str, None]]:
        """
        Pair domain keywords inside a `window`-sentence rolling context.
        Polarity inherits the *window's* dominant sentiment so co-occurrence
        edges aren't blindly positive — a faithful FCM contains both signs.
        Distance-weighted: same-sentence pairs count fully, neighboring
        sentences count proportionally less.
        """
        # Per-pair tallies split by polarity, weighted by proximity
        pos_count: Dict[Tuple[str, str], float] = defaultdict(float)
        neg_count: Dict[Tuple[str, str], float] = defaultdict(float)
        evidence_pool: Dict[Tuple[str, str], List[str]] = defaultdict(list)
        full_text = ' '.join(text_parts)
        sentences = [s for s in re.split(r'(?<=[.!?])\s+', full_text) if s.strip()]

        # Pre-extract domain keywords per sentence
        per_sent: List[List[str]] = []
        for sent in sentences:
            found = list(dict.fromkeys(
                normalize_concept(m.group(0))
                for m in _DOMAIN_KW.finditer(sent)
            ))
            per_sent.append([f for f in found if f and (' ' in f or len(f) >= 6)])

        for i, anchor_sent in enumerate(sentences):
            anchor_keywords = per_sent[i]
            if not anchor_keywords:
                continue
            window_text_parts = [anchor_sent]
            polarity = _sentence_polarity(anchor_sent)

            # Slide window forward up to `window-1` sentences
            for offset in range(window):
                j = i + offset
                if j >= len(sentences):
                    break
                # weight halves for each sentence further away
                proximity = 1.0 / (1 + offset)
                neighbor_kw = per_sent[j]
                if j != i:
                    window_text_parts.append(sentences[j])
                    polarity = polarity if polarity < 0 else _sentence_polarity(sentences[j])

                # All-pairs across anchor + neighbor (skip self-pair, skip backwards
                # within same sentence to avoid duplicate work — but keep both
                # directions since FCM is directed)
                for a in anchor_keywords:
                    for b in neighbor_kw:
                        if a == b:
                            continue
                        # Avoid double-tallying within the same sentence
                        if j == i and a >= b:
                            continue
                        if polarity < 0:
                            neg_count[(a, b)] += proximity
                        else:
                            pos_count[(a, b)] += proximity
                        concepts[a] = None
                        concepts[b] = None
                        if len(evidence_pool[(a, b)]) < 2:
                            evidence_pool[(a, b)].append(anchor_sent.strip())

        combined = {k: pos_count.get(k, 0) + neg_count.get(k, 0)
                    for k in set(pos_count) | set(neg_count)}

        if not combined:
            for item in re.findall(
                r'\b[A-Za-z][A-Za-z\-/]{3,}\b(?:\s+[A-Za-z][A-Za-z\-/]{3,}\b){0,2}',
                full_text,
            )[:12]:
                c = normalize_concept(item)
                if c:
                    concepts[c] = None
            return [], concepts

        max_count = max(combined.values())
        edges: List[Edge] = []
        # Take top-30 (was 15) since the wider window produces more candidates
        for (source, target), count in sorted(combined.items(), key=lambda x: -x[1])[:30]:
            pos = pos_count.get((source, target), 0)
            neg = neg_count.get((source, target), 0)
            is_neg = neg > pos
            magnitude = 0.3 + 0.4 * (count / max_count)
            weight = round(-magnitude if is_neg else magnitude, 3)
            evidence_text = ' || '.join(evidence_pool.get((source, target), [])) \
                            or f'co-occurrence ({count:.1f} weighted, {pos:.1f}+ / {neg:.1f}-)'
            edges.append(Edge(
                source=source,
                target=target,
                weight=weight,
                polarity='negative' if is_neg else 'positive',
                confidence=0.40,
                evidence=evidence_text,
                evidence_doc_id=None,
                edge_type='cooccurrence',
                hops=1,
            ))
        return edges, concepts

    # ------------------------------------------------------------------
    # Transitive closure: derive A→C from A→B + B→C with damped weight
    # ------------------------------------------------------------------
    def _transitive_closure(
        self,
        edges: List[Edge],
        damping: float = 0.7,
        min_weight: float = 0.15,
        max_new: int = 40,
    ) -> List[Edge]:
        """
        Generate 2-hop inferred edges. For each chain A→B→C with no direct
        A→C edge present, add A→C with `damping × w_AB × w_BC`. Marked
        `edge_type='transitive', hops=2` so researchers can filter or
        visually distinguish them from primary evidence.
        """
        # Build outgoing adjacency from existing edges
        out: Dict[str, List[Edge]] = defaultdict(list)
        existing: set = set()
        for e in edges:
            out[e.source].append(e)
            existing.add((e.source, e.target))

        candidates: Dict[Tuple[str, str], Dict] = {}
        for e1 in edges:
            for e2 in out.get(e1.target, []):
                a, c = e1.source, e2.target
                if a == c or (a, c) in existing:
                    continue
                w = damping * e1.weight * e2.weight
                if abs(w) < min_weight:
                    continue
                conf = damping * min(e1.confidence, e2.confidence)
                key = (a, c)
                # Keep the strongest derivation if multiple chains exist
                if key not in candidates or abs(w) > abs(candidates[key]['weight']):
                    candidates[key] = {
                        'weight': w,
                        'confidence': conf,
                        'via': e1.target,
                        'evidence_a': e1.evidence,
                        'evidence_b': e2.evidence,
                    }

        # Sort by absolute weight × confidence and cap
        ranked = sorted(
            candidates.items(),
            key=lambda kv: abs(kv[1]['weight']) * kv[1]['confidence'],
            reverse=True,
        )[:max_new]

        new_edges: List[Edge] = []
        for (a, c), info in ranked:
            new_edges.append(Edge(
                source=a,
                target=c,
                weight=round(info['weight'], 3),
                polarity='positive' if info['weight'] >= 0 else 'negative',
                confidence=round(info['confidence'], 3),
                evidence=f"inferred via {info['via']} :: {info['evidence_a']} → {info['evidence_b']}",
                evidence_doc_id=None,
                edge_type='transitive',
                hops=2,
            ))
        return new_edges


## 5. Graph assembly — `FCM/fcm_graph.py`

Takes the cleaned concept list and edges and packages them into the canonical `FCMMap` (concepts in alphabetical order; adjacency matrix indexed by that order). The `export_network` helper writes JSON, GEXF, two CSVs (adjacency + edge list) and a publication-quality PNG to disk — useful when sharing artefacts with collaborators.


> **Source — `FCM/fcm_graph.py`**

In [ ]:
from __future__ import annotations

import csv
from pathlib import Path
from typing import Dict, List

import networkx as nx

from .schemas import Edge, FCMMap


class FCMGraphBuilder:
    def build(self, query: str, concepts: List[str], edges: List[Edge]) -> FCMMap:
        ordered = sorted(dict.fromkeys(c for c in concepts if c))
        index = {c: i for i, c in enumerate(ordered)}
        matrix = [[0.0 for _ in ordered] for _ in ordered]
        for edge in edges:
            if edge.source in index and edge.target in index:
                matrix[index[edge.source]][index[edge.target]] = float(edge.weight)
        return FCMMap(query=query, concepts=ordered, edges=edges, adjacency_matrix=matrix)

    def export_network(self, fcm_map: FCMMap, outdir: str | Path) -> Dict[str, str]:
        outdir = Path(outdir).resolve()
        outdir.mkdir(parents=True, exist_ok=True)
        json_path = outdir / 'fcm_map.json'
        gexf_path = outdir / 'fcm_map.gexf'
        csv_path = outdir / 'adjacency_matrix.csv'
        png_path = outdir / 'fcm_map.png'
        edge_csv_path = outdir / 'edge_list.csv'
        node_csv_path = outdir / 'node_metrics.csv'

        with json_path.open('w', encoding='utf-8') as fh:
            fh.write(fcm_map.model_dump_json(indent=2))

        with csv_path.open('w', encoding='utf-8', newline='') as fh:
            writer = csv.writer(fh)
            writer.writerow([''] + fcm_map.concepts)
            for concept, row in zip(fcm_map.concepts, fcm_map.adjacency_matrix):
                writer.writerow([concept] + row)

        graph = nx.DiGraph()
        for concept in fcm_map.concepts:
            graph.add_node(concept)
        for edge in fcm_map.edges:
            graph.add_edge(
                edge.source,
                edge.target,
                weight=edge.weight,
                abs_weight=abs(edge.weight),
                sign='positive' if edge.weight >= 0 else 'negative',
                confidence=edge.confidence,
                polarity=edge.polarity,
            )

        if graph.number_of_nodes() > 0:
            in_strength = {
                n: sum(abs(data.get('weight', 0.0)) for _, _, data in graph.in_edges(n, data=True))
                for n in graph.nodes
            }
            out_strength = {
                n: sum(abs(data.get('weight', 0.0)) for _, _, data in graph.out_edges(n, data=True))
                for n in graph.nodes
            }
            total_strength = {n: in_strength[n] + out_strength[n] for n in graph.nodes}
            nx.set_node_attributes(graph, in_strength, 'in_strength')
            nx.set_node_attributes(graph, out_strength, 'out_strength')
            nx.set_node_attributes(graph, total_strength, 'total_strength')

        nx.write_gexf(graph, gexf_path)

        with edge_csv_path.open('w', encoding='utf-8', newline='') as fh:
            writer = csv.writer(fh)
            writer.writerow(['source', 'target', 'weight', 'abs_weight', 'confidence', 'sign'])
            for source, target, data in graph.edges(data=True):
                writer.writerow(
                    [
                        source,
                        target,
                        round(float(data.get('weight', 0.0)), 4),
                        round(float(abs(data.get('weight', 0.0))), 4),
                        round(float(data.get('confidence', 0.0)), 4),
                        data.get('sign', ''),
                    ]
                )

        with node_csv_path.open('w', encoding='utf-8', newline='') as fh:
            writer = csv.writer(fh)
            writer.writerow(['concept', 'in_degree', 'out_degree', 'in_strength', 'out_strength', 'total_strength'])
            for node in graph.nodes:
                writer.writerow(
                    [
                        node,
                        graph.in_degree(node),
                        graph.out_degree(node),
                        round(float(graph.nodes[node].get('in_strength', 0.0)), 4),
                        round(float(graph.nodes[node].get('out_strength', 0.0)), 4),
                        round(float(graph.nodes[node].get('total_strength', 0.0)), 4),
                    ]
                )

        try:
            import math
            import textwrap
            import matplotlib
            matplotlib.use('Agg')
            import matplotlib.pyplot as plt
            import matplotlib.patches as mpatches
            from matplotlib.lines import Line2D
            from matplotlib.patches import FancyArrowPatch

            if graph.number_of_nodes() > 0:
                nodes_g = list(graph.nodes())
                ng = len(nodes_g)

                # Node strength metrics
                node_strength = [float(graph.nodes[nd].get('total_strength', 0.0)) for nd in nodes_g]
                min_s = min(node_strength) if node_strength else 0.0
                max_s = max(node_strength) if node_strength else 1.0

                def _ns(v: float) -> float:
                    if max_s == min_s:
                        return 2000.0
                    return 1800.0 + ((v - min_s) / (max_s - min_s)) * 3800.0

                sizes_g = [_ns(s) for s in node_strength]

                # Kamada-Kawai layout
                try:
                    pos_g = nx.kamada_kawai_layout(graph, weight=None)
                except Exception:
                    pos_g = nx.spring_layout(graph, seed=42,
                                             k=2.8 / max(1, ng ** 0.5),
                                             iterations=500)

                fig_g, ax_g = plt.subplots(figsize=(18, 13))
                ax_g.set_facecolor('#F7F9FC')
                fig_g.patch.set_facecolor('#F7F9FC')
                ax_g.axis('off')
                ax_g.set_xlim(-1.45, 1.45)
                ax_g.set_ylim(-1.45, 1.45)

                cx_g = sum(p[0] for p in pos_g.values()) / ng
                cy_g = sum(p[1] for p in pos_g.values()) / ng

                C_POS_G = '#1A6B3C'
                C_NEG_G = '#B03A2E'

                positive_edges = [(u, v, d) for u, v, d in graph.edges(data=True)
                                  if float(d.get('weight', 0.0)) >= 0]
                negative_edges = [(u, v, d) for u, v, d in graph.edges(data=True)
                                  if float(d.get('weight', 0.0)) < 0]

                drawn_g: set = set()

                def _shrink_g(nd: str) -> float:
                    idx = nodes_g.index(nd) if nd in nodes_g else 0
                    return math.sqrt(sizes_g[idx]) / 2.0 + 5

                def _arc_mp(p1, p2, rad):
                    mx, my = (p1[0]+p2[0])/2, (p1[1]+p2[1])/2
                    dx, dy = p2[0]-p1[0], p2[1]-p1[1]
                    length = (dx**2 + dy**2)**0.5
                    if length < 1e-9:
                        return mx, my
                    px, py = -dy/length, dx/length
                    return mx + px*rad*length*0.5, my + py*rad*length*0.5

                def _draw_g(u, v, d, col, rad, z):
                    w  = float(d.get('weight', 0.0))
                    lw = 1.4 + abs(w) * 3.0
                    r  = -rad if (v, u) in drawn_g else rad
                    drawn_g.add((u, v))
                    ax_g.add_patch(FancyArrowPatch(
                        posA=pos_g[u], posB=pos_g[v],
                        connectionstyle=f'arc3,rad={r}',
                        arrowstyle='-|>', mutation_scale=24,
                        color=col, linewidth=lw, alpha=0.90,
                        shrinkA=_shrink_g(u), shrinkB=_shrink_g(v), zorder=z,
                    ))
                    lx, ly = _arc_mp(pos_g[u], pos_g[v], r)
                    badge = f'+{abs(w):.2f}' if w >= 0 else f'\u2212{abs(w):.2f}'
                    ax_g.text(lx, ly, badge, fontsize=9, color='white',
                              ha='center', va='center', fontweight='bold',
                              bbox=dict(facecolor=C_POS_G if w >= 0 else C_NEG_G,
                                        alpha=0.95, edgecolor='none',
                                        boxstyle='round,pad=0.24'),
                              zorder=z + 2)

                for u, v, d in positive_edges:
                    _draw_g(u, v, d, C_POS_G,  0.20, 2)
                for u, v, d in negative_edges:
                    _draw_g(u, v, d, C_NEG_G, -0.20, 3)

                # Draw nodes + external labels
                for idx, nd in enumerate(nodes_g):
                    x_g, y_g = pos_g[nd]
                    sz_g = sizes_g[idx]
                    ax_g.scatter(x_g, y_g, s=sz_g * 1.65,
                                 c='none', edgecolors='#2C3E50',
                                 linewidths=3.5, zorder=4)
                    ax_g.scatter(x_g, y_g, s=sz_g, c='#2C6E9E',
                                 alpha=0.88, edgecolors='white',
                                 linewidths=2.0, zorder=5)
                    # External label
                    dx_g = x_g - cx_g
                    dy_g = y_g - cy_g
                    dist_g = math.sqrt(dx_g**2 + dy_g**2) + 1e-9
                    ux_g, uy_g = dx_g/dist_g, dy_g/dist_g
                    off_g = math.sqrt(sz_g)/130.0 + 0.13
                    tx_g  = x_g + ux_g * off_g
                    ty_g  = y_g + uy_g * off_g
                    label_g = '\n'.join(textwrap.wrap(nd, width=14))
                    rel_g   = (node_strength[idx] - min_s) / (max_s - min_s + 1e-9)
                    fsize_g = max(9, int(9 + 4 * rel_g))
                    ax_g.plot([x_g, tx_g], [y_g, ty_g],
                              color='#2C6E9E', lw=0.9, alpha=0.5, zorder=4)
                    ax_g.text(tx_g, ty_g, label_g,
                              ha='center', va='center',
                              fontsize=fsize_g, fontweight='bold',
                              color='#0d1b2a', zorder=7,
                              multialignment='center', linespacing=1.3,
                              bbox=dict(facecolor='white', alpha=0.92,
                                        edgecolor='#2C6E9E', linewidth=1.4,
                                        boxstyle='round,pad=0.30'))

                legend_g = [
                    Line2D([0], [0], color=C_POS_G, linewidth=3.5,
                           label=f'Positive causal link  (n={len(positive_edges)})'),
                    Line2D([0], [0], color=C_NEG_G, linewidth=3.5,
                           label=f'Negative causal link  (n={len(negative_edges)})'),
                ]
                leg_g = ax_g.legend(handles=legend_g, loc='upper right',
                                    framealpha=0.96, facecolor='white',
                                    edgecolor='#cccccc', fontsize=10.5,
                                    title='Edge Types', title_fontsize=11.5)
                leg_g.get_title().set_fontweight('bold')
                leg_g.get_title().set_color('#013056')

                ax_g.set_title(
                    'Gulf of Mexico FEI  \u00b7  Fuzzy Cognitive Map\n'
                    'Signed Causal Ecosystem Network',
                    fontsize=14, fontweight='bold', color='#013056',
                    pad=18, linespacing=1.7,
                )
                plt.figtext(
                    0.5, 0.008,
                    f'Nodes={graph.number_of_nodes()}  |  Edges={graph.number_of_edges()}  |  '
                    'Node size = total causal strength  |  '
                    'Green = positive  |  Red = negative  |  Layout: Kamada-Kawai',
                    ha='center', fontsize=9, color='#5a7099', style='italic',
                )
                plt.tight_layout(rect=[0, 0.03, 1, 0.97])
                plt.savefig(png_path, dpi=240, bbox_inches='tight', facecolor='#F7F9FC')
                plt.close()
        except Exception:
            pass

        return {
            'json': str(json_path),
            'gexf': str(gexf_path),
            'csv': str(csv_path),
            'png': str(png_path) if png_path.exists() else '',
            'edge_csv': str(edge_csv_path),
            'node_csv': str(node_csv_path),
        }


## 6. High-level orchestrator — `src/FCM.py`

Wraps `CausalExtractor + FCMGraphBuilder` behind two convenience methods used by the FastAPI app:

- `build_fcm(text, query, num_results)` — the standard path used by `/query` and `/upload-files` on plain text.
- `build_fcm_from_matrix(concepts, matrix)` — bypass extraction when the user uploads an adjacency CSV / XLSX.

It also owns the publication-quality renderer (`plot_fcm`) with Kamada–Kawai layout, betweenness-centrality node sizing, greedy modularity colouring, role rings (transmitter / receiver / ordinary / isolated) and an embedded statistics panel.


> **Source — `src/FCM.py`**

In [ ]:
import math
import os
import textwrap
import logging
from typing import List, Dict, Tuple, Optional

import networkx as nx
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from FCM.extractor import CausalExtractor
from FCM.fcm_graph import FCMGraphBuilder
from FCM.schemas import Edge, FCMMap
from FCM.categorizer import get_summary_theme

MAX_CONCEPTS = 12
EDGE_THRESHOLD = 0.2

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ── Helpers ───────────────────────────────────────────────────────────────────

def _wrap_label(text: str, width: int = 14) -> str:
    return "\n".join(textwrap.wrap(text, width=width))


def _arc_midpoint(p1: tuple, p2: tuple, rad: float) -> tuple:
    mx = (p1[0] + p2[0]) / 2
    my = (p1[1] + p2[1]) / 2
    dx, dy = p2[0] - p1[0], p2[1] - p1[1]
    length = (dx ** 2 + dy ** 2) ** 0.5
    if length < 1e-9:
        return mx, my
    px, py = -dy / length, dx / length
    return mx + px * rad * length * 0.5, my + py * rad * length * 0.5


# ── Main class ────────────────────────────────────────────────────────────────

class FCMGenerator:
    def __init__(
        self,
        max_concepts: int = MAX_CONCEPTS,
        edge_threshold: float = EDGE_THRESHOLD,
        max_edges: Optional[int] = None,
        directed: bool = True,
    ):
        self.max_concepts  = max_concepts
        self.edge_threshold = edge_threshold
        self.max_edges     = max_edges
        self.directed      = directed

        self._extractor     = CausalExtractor()
        self._graph_builder = FCMGraphBuilder()

    # ------------------------------------------------------------------
    def _extract_edges(self, text: str, num_results: int = 0) -> Tuple[List[str], List[Edge]]:
        chunks = [{"text": text, "score": 1.0, "doc_id": "context"}]
        concepts, edges = self._extractor.extract(chunks)
        if num_results:
            edges = edges[:num_results]
        return concepts, edges

    def _build_graph(self, query: str, edges: List[Edge]) -> Tuple[nx.DiGraph, "FCMMap"]:
        # Derive the concept set from the edges so the adjacency matrix is
        # populated (passing [] would leave the matrix empty).
        edge_concepts: List[str] = []
        seen: set = set()
        for e in edges:
            for node in (e.source, e.target):
                if node and node not in seen:
                    seen.add(node)
                    edge_concepts.append(node)

        fcm_map: FCMMap = self._graph_builder.build(query, edge_concepts, edges)
        G = nx.DiGraph() if self.directed else nx.Graph()
        # Carry richer attrs (edge_type, hops, confidence) so the plot can
        # style transitive edges differently, and the UI can show evidence.
        edge_attrs: Dict[Tuple[str, str], Edge] = {(e.source, e.target): e for e in edges}
        for edge in fcm_map.edges:
            src_edge = edge_attrs.get((edge.source, edge.target), edge)
            G.add_edge(
                edge.source, edge.target,
                weight     = edge.weight,
                polarity   = edge.polarity,
                edge_type  = getattr(src_edge, 'edge_type', 'pattern'),
                hops       = getattr(src_edge, 'hops', 1),
                confidence = getattr(src_edge, 'confidence', 0.7),
                evidence   = getattr(src_edge, 'evidence', ''),
            )
        return G, fcm_map

    # ------------------------------------------------------------------
    def build_fcm(self, text: str, query: str, num_results: int = 6) -> Tuple[nx.DiGraph, Dict]:
        _, all_edges = self._extract_edges(text, 0)
        lim_edges    = all_edges[:num_results] if num_results else all_edges

        G, fcm_map = self._build_graph(query, lim_edges)

        num_n = G.number_of_nodes()
        # Build a lookup for the rich edge metadata (Edge objects)
        rich_lookup: Dict[Tuple[str, str], Edge] = {(e.source, e.target): e for e in lim_edges}
        edges_meta = []
        for e in fcm_map.edges:
            src = rich_lookup.get((e.source, e.target), e)
            edges_meta.append({
                "source":     e.source,
                "target":     e.target,
                "weight":     round(float(e.weight), 3),
                "polarity":   e.polarity,
                "confidence": round(float(getattr(src, 'confidence', 0.7)), 3),
                "edge_type":  getattr(src, 'edge_type', 'pattern'),
                "hops":       int(getattr(src, 'hops', 1)),
                "evidence":   getattr(src, 'evidence', ''),
                "evidence_doc_id": getattr(src, 'evidence_doc_id', None),
            })
        type_tally = {}
        for em in edges_meta:
            type_tally[em["edge_type"]] = type_tally.get(em["edge_type"], 0) + 1
        details = {
            "nodes": fcm_map.concepts,
            "edges": [(e.source, e.target, e.weight) for e in fcm_map.edges],
            "edges_meta": edges_meta,
            "adjacency_matrix": fcm_map.adjacency_matrix,
            "stats": {
                "num_nodes":      num_n,
                "num_edges":      G.number_of_edges(),
                "average_degree": sum(dict(G.degree()).values()) / num_n if num_n else 0,
                "density":        nx.density(G),
                "edge_types":     type_tally,
            },
        }
        logger.info("Built FCM with %d nodes and %d edges (%s).",
                    num_n, G.number_of_edges(), type_tally)
        return G, details

    # ------------------------------------------------------------------
    def build_fcm_from_matrix(
        self,
        concepts: List[str],
        matrix: List[List[float]],
        query: str = "Uploaded Matrix",
    ) -> Tuple[nx.DiGraph, Dict]:
        """Build FCM graph directly from an adjacency matrix (e.g. uploaded CSV)."""
        G = nx.DiGraph() if self.directed else nx.Graph()
        edges_list: List[Tuple] = []

        for i, src in enumerate(concepts):
            for j, tgt in enumerate(concepts):
                if i == j:
                    continue
                try:
                    w = float(matrix[i][j])
                except (IndexError, TypeError, ValueError):
                    w = 0.0
                if w != 0.0:
                    G.add_edge(src, tgt, weight=w,
                               polarity="positive" if w > 0 else "negative")
                    edges_list.append((src, tgt, w))

        num_n = G.number_of_nodes()
        edges_meta = [
            {
                "source": s, "target": t, "weight": round(float(w), 3),
                "polarity": "positive" if w >= 0 else "negative",
                "confidence": 0.95, "edge_type": "matrix", "hops": 1,
                "evidence": "loaded from uploaded adjacency matrix",
                "evidence_doc_id": None,
            }
            for s, t, w in edges_list
        ]
        details = {
            "nodes": concepts,
            "edges": edges_list,
            "edges_meta": edges_meta,
            "adjacency_matrix": matrix,
            "stats": {
                "num_nodes":      num_n,
                "num_edges":      G.number_of_edges(),
                "average_degree": sum(dict(G.degree()).values()) / num_n if num_n else 0,
                "density":        nx.density(G),
                "edge_types":     {"matrix": len(edges_meta)},
            },
        }
        logger.info("Built FCM from matrix: %d nodes, %d edges.", num_n, len(edges_list))
        return G, details

    def format_edges_as_results(self, relations: List[Dict], limit: Optional[int] = None) -> List[Dict]:
        results = []
        for r in (relations[:limit] if limit else relations):
            try:
                results.append({
                    "result":   r.get("id", len(results) + 1),
                    "cause":    r.get("cause", "unknown"),
                    "effect":   r.get("effect", "unknown"),
                    "polarity": r.get("polarity", "+"),
                    "weight":   round(float(r.get("weight", 0.5)), 2),
                })
            except (KeyError, ValueError) as exc:
                logger.warning("Skipping invalid relation: %s — %s", r, exc)
        return results

    # ------------------------------------------------------------------
    # Visualization
    # ------------------------------------------------------------------

    def plot_fcm(
        self,
        G: nx.DiGraph,
        filename: str = "fcm_graph",
        query: str = "",
    ) -> Optional[str]:
        """
        Publication-quality FCM — all labels guaranteed visible.

        Root causes of invisible text that are fixed here:
          1. No manual ax.set_xlim/ylim — labels were clipped outside the bounds
          2. Correct node-radius → data-unit conversion for label offsets
          3. Minimum font size 13 pt; weight badges 11 pt; stats panel 10 pt
          4. bbox_inches="tight" captures all text artists outside the axis
        """
        if not G.nodes:
            return None

        from matplotlib.lines import Line2D
        from matplotlib.patches import FancyArrowPatch

        static_dir = os.path.join(os.getcwd(), "static")
        os.makedirs(static_dir, exist_ok=True)
        png_path = os.path.join(static_dir, f"{filename}.png")

        nodes = list(G.nodes())
        n     = len(nodes)

        # ── 1. FCM node metrics ───────────────────────────────────────
        in_str: Dict[str, float] = {
            nd: sum(abs(float(d.get("weight", 0))) for _, _, d in G.in_edges(nd, data=True))
            for nd in nodes
        }
        out_str: Dict[str, float] = {
            nd: sum(abs(float(d.get("weight", 0))) for _, _, d in G.out_edges(nd, data=True))
            for nd in nodes
        }
        total_str: Dict[str, float] = {nd: in_str[nd] + out_str[nd] for nd in nodes}

        # ── 2. FCM node role classification ──────────────────────────
        def _role(nd: str) -> str:
            o, i = out_str[nd], in_str[nd]
            if o == 0 and i == 0:
                return "Isolated"
            if i < 1e-9 or (o > 0 and o / (i + 1e-9) > 2.5):
                return "Transmitter"
            if o < 1e-9 or (i > 0 and i / (o + 1e-9) > 2.5):
                return "Receiver"
            return "Ordinary"

        roles: Dict[str, str] = {nd: _role(nd) for nd in nodes}

        ROLE_RING = {
            "Transmitter": "#E67E22",
            "Receiver":    "#2471A3",
            "Ordinary":    "#1E8449",
            "Isolated":    "#808B96",
        }
        ROLE_LW = {"Transmitter": 5.5, "Receiver": 5.0, "Ordinary": 4.5, "Isolated": 2.5}

        # ── 3. Centrality (importance) ────────────────────────────────
        try:
            btw = nx.betweenness_centrality(G, normalized=True)
        except Exception:
            btw = {nd: 0.5 for nd in nodes}
        deg = nx.degree_centrality(G)
        importance: Dict[str, float] = {
            nd: 0.55 * btw.get(nd, 0) + 0.45 * deg.get(nd, 0) for nd in nodes
        }
        mx_imp = max(importance.values()) if any(v > 0 for v in importance.values()) else 1.0

        def _rel(nd: str) -> float:
            return importance[nd] / max(mx_imp, 1e-9)

        # Node scatter size 700 → 2 200 pts²  (smaller = less overlap)
        def _size(nd: str) -> float:
            return 700 + 1500 * _rel(nd)

        # Arrow shrink in display pts (keeps arrowheads off the circle surface)
        def _shrink(nd: str) -> float:
            return math.sqrt(_size(nd)) / 2.0 + 4

        # ── 4. Community / module detection (greedy modularity) ─────
        try:
            from networkx.algorithms.community import greedy_modularity_communities
            UG    = G.to_undirected()
            comms = list(greedy_modularity_communities(UG)) if UG.number_of_edges() > 0 \
                    else [frozenset(nodes)]
        except Exception:
            comms = [frozenset(nodes)]
        if not comms:
            comms = [frozenset(nodes)]

        node_comm: Dict[str, int] = {nd: i for i, c in enumerate(comms) for nd in c}

        # Okabe-Ito colorblind-safe palette (matches reference design)
        PALETTE = [
            "#0077BB", "#EE7733", "#009988", "#CC3311", "#33BBEE",
            "#EE3377", "#BBBBBB", "#117733", "#44BB99", "#AAAA00",
            "#99DDFF", "#FFAABB", "#332288", "#882255",
        ]

        def _ccolor(nd: str) -> str:
            return PALETTE[node_comm.get(nd, 0) % len(PALETTE)]

        # ── 5. Layout + mandatory spacing enforcement ─────────────────
        # Kamada-Kawai gives structured topology but packs dense graphs
        # too tightly.  We rescale the raw positions so every adjacent
        # pair of nodes is visually separated, then enforce a hard
        # minimum-distance floor to eliminate residual overlaps.
        try:
            pos = nx.kamada_kawai_layout(G, weight=None)
        except Exception:
            try:
                pos = nx.spring_layout(
                    G, k=4.0 / math.sqrt(max(n, 1)), iterations=600, seed=42
                )
            except Exception:
                pos = nx.circular_layout(G)

        # Rescale: spread positions so the layout spans ±target_half
        # target_half grows with node count to keep per-node area constant
        target_half = max(1.8, math.sqrt(n) * 0.45)
        xs = [v[0] for v in pos.values()]
        ys = [v[1] for v in pos.values()]
        x_mid = (max(xs) + min(xs)) / 2
        y_mid = (max(ys) + min(ys)) / 2
        x_span = max(max(xs) - min(xs), 1e-9)
        y_span = max(max(ys) - min(ys), 1e-9)
        pos = {
            nd: (
                (x - x_mid) / x_span * target_half * 2,
                (y - y_mid) / y_span * target_half * 2,
            )
            for nd, (x, y) in pos.items()
        }

        # Hard minimum-distance floor: push overlapping nodes apart
        min_sep = 0.55   # data units — must exceed sum of two max node radii
        _moved  = True
        _iters  = 0
        while _moved and _iters < 30:
            _moved = False
            _iters += 1
            node_list = list(pos.keys())
            for i in range(len(node_list)):
                for j in range(i + 1, len(node_list)):
                    na, nb = node_list[i], node_list[j]
                    ax_, ay_ = pos[na]
                    bx_, by_ = pos[nb]
                    dx_, dy_ = bx_ - ax_, by_ - ay_
                    dist_   = math.sqrt(dx_ ** 2 + dy_ ** 2) + 1e-12
                    if dist_ < min_sep:
                        push    = (min_sep - dist_) / 2.0
                        ux_, uy_ = dx_ / dist_, dy_ / dist_
                        pos[na] = (ax_ - ux_ * push, ay_ - uy_ * push)
                        pos[nb] = (bx_ + ux_ * push, by_ + uy_ * push)
                        _moved  = True

        # Graph centroid for radial label direction
        cx_all = sum(p[0] for p in pos.values()) / n
        cy_all = sum(p[1] for p in pos.values()) / n

        # ── 6. Figure — DO NOT set explicit xlim/ylim ─────────────────
        # Reason: explicit limits clip external labels placed outside
        # the data range.  bbox_inches="tight" captures all artists.
        FIG_W, FIG_H = 28, 20
        fig, ax = plt.subplots(figsize=(FIG_W, FIG_H))
        ax.set_facecolor("#F7F9FC")
        fig.patch.set_facecolor("#F7F9FC")
        ax.axis("off")

        # ── 7. Edges ─────────────────────────────────────────────────
        C_POS = "#1A6B3C"
        C_NEG = "#B03A2E"

        all_edges   = list(G.edges(data=True))
        pos_edges   = [(u, v, d) for u, v, d in all_edges if float(d.get("weight", 0)) >= 0]
        neg_edges   = [(u, v, d) for u, v, d in all_edges if float(d.get("weight", 0)) < 0]
        drawn_pairs: set = set()

        logger.info("FCM plot: %d positive, %d negative edges",
                    len(pos_edges), len(neg_edges))

        def _draw_edge(u: str, v: str, d: dict,
                       base_color: str, base_rad: float, z: int) -> None:
            w   = float(d.get("weight", 0))
            lw  = 1.4 + abs(w) * 3.2
            rad = -base_rad if (v, u) in drawn_pairs else base_rad
            drawn_pairs.add((u, v))

            etype = d.get("edge_type", "pattern")
            conf  = float(d.get("confidence", 0.7))
            # Visual differentiation: transitive (inferred) edges get a dashed
            # stroke and faded alpha so reviewers can tell them apart from
            # primary, evidence-backed (pattern/cooccurrence/matrix) edges.
            if etype == "transitive":
                style = (0, (5, 3))
                edge_alpha = 0.55
                lw = max(1.0, lw * 0.75)
            else:
                style = "solid"
                edge_alpha = 0.65 + 0.30 * min(1.0, conf)

            ax.add_patch(FancyArrowPatch(
                posA=pos[u], posB=pos[v],
                connectionstyle=f"arc3,rad={rad}",
                arrowstyle="-|>",
                mutation_scale=26,
                color=base_color,
                linewidth=lw,
                alpha=edge_alpha,
                linestyle=style,
                shrinkA=_shrink(u),
                shrinkB=_shrink(v),
                zorder=z,
            ))

            # Weight badge — white bold text on solid color pill
            lx, ly = _arc_midpoint(pos[u], pos[v], rad)
            badge  = f"+{abs(w):.2f}" if w >= 0 else f"-{abs(w):.2f}"
            ax.text(
                lx, ly, badge,
                fontsize=11,            # ← 11 pt ensures readability
                color="white",
                ha="center", va="center", fontweight="bold",
                bbox=dict(
                    facecolor=C_POS if w >= 0 else C_NEG,
                    alpha=0.97, edgecolor="none",
                    boxstyle="round,pad=0.28",
                ),
                zorder=z + 2,
            )

        for u, v, d in pos_edges:
            _draw_edge(u, v, d, C_POS,  0.22, 2)
        for u, v, d in neg_edges:
            _draw_edge(u, v, d, C_NEG, -0.22, 3)

        # ── 8. Nodes + external labels ────────────────────────────────
        # After rescaling, positions span ±target_half ≈ 1.8–4.
        # Figure: 28 in, plot width ≈ 28*0.64 = 17.9 in (rect leaves 14% left, 22% right).
        # Data range = 2 * target_half.
        # pts/data-unit = (17.9 / (2*target_half)) * 72
        data_range = target_half * 2
        SCALE = (17.9 / data_range) * 72      # pts per data unit

        for nd in nodes:
            x, y  = pos[nd]
            color = _ccolor(nd)
            sz    = _size(nd)
            rel   = _rel(nd)
            role  = roles[nd]

            # Role ring (outer)
            ax.scatter(x, y, s=sz * 1.80,
                       c="none", edgecolors=ROLE_RING[role],
                       linewidths=ROLE_LW[role], zorder=4)
            # Main circle
            ax.scatter(x, y, s=sz, c=color, alpha=0.92,
                       edgecolors="white", linewidths=2.5, zorder=5)

            # Radial direction: outward from graph centroid
            dx, dy = x - cx_all, y - cy_all
            dist   = math.sqrt(dx ** 2 + dy ** 2) + 1e-9
            ux, uy = dx / dist, dy / dist

            # Visual node radius in data units, plus clearance gap
            r_data  = math.sqrt(sz / math.pi) / SCALE
            lbl_off = r_data + 0.18          # 0.18 data-unit gap ensures label clears ring
            tx, ty  = x + ux * lbl_off, y + uy * lbl_off

            fsize = max(11, int(11 + 4 * rel))

            # Wrap at 16 chars so no label exceeds 3 lines
            label = _wrap_label(nd, width=16)

            # Connector: starts at node surface, ends at label center
            ax.plot(
                [x + ux * r_data * 1.05, tx],
                [y + uy * r_data * 1.05, ty],
                color=color, lw=1.6, alpha=0.60, zorder=4,
                solid_capstyle="round",
            )

            ax.text(
                tx, ty, label,
                ha="center", va="center",
                fontsize=fsize, fontweight="bold",
                color="#0d1b2a",
                zorder=7,
                multialignment="center",
                linespacing=1.40,
                bbox=dict(
                    facecolor="white", alpha=0.96,
                    edgecolor=color, linewidth=1.8,
                    boxstyle="round,pad=0.36",
                ),
            )

        # ── 9. Legend ─────────────────────────────────────────────────
        legend_handles: list = []

        for i, comm in enumerate(comms):
            color  = PALETTE[i % len(PALETTE)]
            rep_nd = max(comm, key=lambda nd: importance.get(nd, 0))
            rep    = (rep_nd[:26] + "...") if len(rep_nd) > 26 else rep_nd
            legend_handles.append(
                mpatches.Patch(facecolor=color, edgecolor="white",
                               linewidth=1.5,
                               label=f"Module {i + 1}  -  {rep}")
            )

        legend_handles.append(
            mpatches.Patch(facecolor="none", edgecolor="none", label="")
        )
        n_transitive = sum(
            1 for _, _, d in all_edges
            if d.get("edge_type") == "transitive"
        )
        legend_handles += [
            Line2D([0], [0], color=C_POS, linewidth=4.5,
                   label=f"Positive causal link   n = {len(pos_edges)}"),
            Line2D([0], [0], color=C_NEG, linewidth=4.5,
                   label=f"Negative causal link   n = {len(neg_edges)}"),
        ]
        if n_transitive:
            legend_handles.append(
                Line2D([0], [0], color="#5a7099", linewidth=2.5,
                       linestyle=(0, (5, 3)),
                       label=f"Inferred (2-hop)      n = {n_transitive}")
            )
        legend_handles.append(
            mpatches.Patch(facecolor="none", edgecolor="none", label="")
        )
        for role, rcolor in ROLE_RING.items():
            rc = sum(1 for v in roles.values() if v == role)
            if rc > 0:
                legend_handles.append(
                    Line2D([0], [0], marker="o", color="none",
                           markerfacecolor="none",
                           markeredgecolor=rcolor,
                           markeredgewidth=4.5, markersize=15,
                           label=f"{role}   n = {rc}")
                )

        # Place legend outside the axes on the right side so it never
        # overlaps node labels.  bbox_to_anchor=(1.01, 1.0) anchors to the
        # top-right corner of the axes bounding box; loc="upper left" means
        # the legend's own upper-left corner sits at that anchor point.
        leg = ax.legend(
            handles=legend_handles,
            loc="upper left",
            bbox_to_anchor=(1.01, 1.0),
            framealpha=0.97,
            facecolor="white",
            edgecolor="#cccccc",
            labelcolor="#0d1b2a",
            fontsize=12,
            title="Modules  |  Edge Types  |  Node Roles",
            title_fontsize=13,
            borderpad=1.3,
            labelspacing=0.90,
        )
        leg.get_title().set_fontweight("bold")
        leg.get_title().set_color("#013056")
        leg.set_zorder(20)   # draw above every node label (z=7) and edge badge (z=5)

        # ── 10. Statistics panel ──────────────────────────────────────
        density    = nx.density(G)
        avg_deg    = sum(dict(G.degree()).values()) / n if n > 0 else 0.0
        top5       = sorted(total_str.items(), key=lambda x: -x[1])[:5]
        role_counts = {r: sum(1 for v in roles.values() if v == r) for r in ROLE_RING}

        top5_lines = "\n".join(
            f"  {idx + 1}. {nd[:20]:<20}  {v:.3f}"
            for idx, (nd, v) in enumerate(top5)
        )
        sep = "-" * 30
        stats_block = (
            f"  Graph Statistics\n"
            f"  {sep}\n"
            f"  Concepts (nodes) : {n}\n"
            f"  Causal edges     : {G.number_of_edges()}\n"
            f"  Positive (+)     : {len(pos_edges)}\n"
            f"  Negative (-)     : {len(neg_edges)}\n"
            f"  Graph density    : {density:.4f}\n"
            f"  Avg degree       : {avg_deg:.2f}\n"
            f"  Modules          : {len(comms)}\n"
            f"\n"
            f"  Node Roles\n"
            f"  {sep}\n"
            f"  Transmitters     : {role_counts.get('Transmitter', 0)}\n"
            f"  Receivers        : {role_counts.get('Receiver', 0)}\n"
            f"  Ordinary         : {role_counts.get('Ordinary', 0)}\n"
            f"  Isolated         : {role_counts.get('Isolated', 0)}\n"
            f"\n"
            f"  Top Concepts (Total Strength)\n"
            f"  {sep}\n"
            f"{top5_lines}"
        )

        fig.text(
            0.013, 0.50,
            stats_block,
            fontsize=10,            # ← 10 pt — readable on web and print
            color="#0d1b2a",
            fontfamily="monospace",
            va="center", ha="left",
            bbox=dict(facecolor="white", alpha=0.95,
                      edgecolor="#aaaaaa", linewidth=1.0,
                      boxstyle="round,pad=0.6"),
            transform=fig.transFigure,
        )

        # ── 11. Title ─────────────────────────────────────────────────
        q_line = (
            f'\nQuery: "{query[:85]}{"..." if len(query) > 85 else ""}"'
            if query else ""
        )
        ax.set_title(
            f"Gulf of Mexico  |  Fisheries Ecosystem Initiative (FEI)\n"
            f"Fuzzy Cognitive Map  -  Signed Causal Ecosystem Network"
            f"{q_line}",
            fontsize=17, fontweight="bold", color="#013056",
            pad=26, linespacing=1.85,
        )

        # ── 12. Methodological footnote ───────────────────────────────
        fig.text(
            0.5, 0.005,
            "Node size proportional to betweenness centrality  "
            "|  Border ring = FCM role: orange=Transmitter  blue=Receiver  "
            "green=Ordinary  gray=Isolated  "
            "|  Fill = semantic module (greedy modularity)  "
            "|  Arrow width proportional to |weight|  "
            "|  Weight scale [-1.0, +1.0]  "
            "|  Layout: Kamada-Kawai energy minimization",
            ha="center", va="bottom",
            fontsize=9.5, color="#5a7099", style="italic",
        )

        # rect leaves room: left=14% (stats panel), right=78% (legend outside)
        plt.tight_layout(rect=[0.14, 0.03, 0.78, 0.96])
        plt.savefig(
            png_path, dpi=200,
            bbox_inches="tight",    # captures legend + labels outside axis bounds
            facecolor="#F7F9FC", edgecolor="none",
        )
        plt.close(fig)

        logger.info("FCM saved -> %s  (%d nodes, %d modules, %d+/%d-)",
                    png_path, n, len(comms), len(pos_edges), len(neg_edges))
        return f"/static/{filename}.png"

    # ------------------------------------------------------------------
    # Summary / Thematic Aggregation (mirrors R summary_mat_final pipeline)
    # ------------------------------------------------------------------

    def aggregate_summary_graph(
        self,
        G: nx.DiGraph,
        weight_threshold: float = 0.5,
    ) -> nx.DiGraph:
        """Collapse a full FCM graph into thematic summary nodes.

        Each original node is mapped to its SummaryTheme via FCM.categorizer.
        Edge weights between same-theme pairs are summed (matching the R
        rowsum/colsum consolidation). Self-loops and edges whose absolute
        aggregated weight is <= weight_threshold are dropped.
        """
        if G is None or G.number_of_nodes() == 0:
            return nx.DiGraph()

        sum_weights: Dict[Tuple[str, str], float] = {}
        for u, v, data in G.edges(data=True):
            s = get_summary_theme(u)
            t = get_summary_theme(v)
            if not s or not t or s == t:
                continue
            w = float(data.get("weight", 0.0))
            sum_weights[(s, t)] = sum_weights.get((s, t), 0.0) + w

        H = nx.DiGraph()
        for (s, t), w in sum_weights.items():
            if abs(w) <= weight_threshold:
                continue
            H.add_edge(s, t, weight=w,
                       polarity="positive" if w >= 0 else "negative")
        return H

    def plot_summary_fcm(
        self,
        G: nx.DiGraph,
        filename: str = "fcm_summary",
        query: str = "",
        weight_threshold: float = 0.5,
    ) -> Optional[str]:
        """Build the summary-theme graph and render it with the same styling."""
        H = self.aggregate_summary_graph(G, weight_threshold=weight_threshold)
        if H.number_of_nodes() == 0:
            return None
        return self.plot_fcm(H, filename=filename, query=query)


## 7. Thematic categorisation — `FCM/categorizer.py`

Implements the FEP (Fishery Ecosystem Plan) colour scheme and the `SummaryTheme` rules used to collapse fine-grained nodes into policy-relevant super-nodes (e.g. *Storms and Flooding*, *Water Quality and Pollution*). Mirrors the original R `case_when` semantics so the Python output stays consistent with the upstream R pipeline.


> **Source — `FCM/categorizer.py`**

In [ ]:
from __future__ import annotations

import re
from typing import Dict, List, Tuple

# Thematic FEP colour scheme (exact hex values from client R script)
FEP_COLORS: Dict[str, str] = {
    "Habitat":             "#228B22",
    "Fishery":             "#1E90FF",
    "Biotic":              "#32CD32",
    "Reef Fish":           "#FFD700",
    "Structures":          "#808080",
    "Environmental":       "#FF8C00",
    "Social and Economic": "#9370DB",
    "Management":          "#E6194B",
    "Science and Data":    "#8B4513",
    "Land-Based Drivers":  "#A52A2A",
    "Other":               "#D3D3D3",
}

# Ordered category rules — first match wins (mirrors R case_when semantics)
_CATEGORY_RULES: List[Tuple[str, str]] = [
    ("Habitat",             r"habitat|estuary|marsh|grass|nursery|restoration|dredging|diversion"),
    ("Fishery",             r"fisher|fishery|effort|harvest|catch|discard|mortality|ifq|quota|import|bycatch|menhaden|shrimp|oyster"),
    ("Reef Fish",           r"snapper|amberjack|trigger|reef\s*fish|spawning"),
    ("Biotic",              r"biotic|biodiversity|shark|dolphin|mammal|turtle|sturgeon|manatee|birds|trophic|bait|crab"),
    ("Structures",          r"reef|rig|platform|structure|windfarm|concrete|infrastructure"),
    ("Environmental",       r"climate|temp|salinity|oxygen|hypoxia|acidification|weather|storm|hurricane|sea-level|tide"),
    ("Social and Economic", r"social|economic|well-being|price|cost|demand|tourism|angler|fleet|access|perception|jobs|fuel|aging|fleets"),
    ("Management",          r"management|council|noaa|regulation|season|limit|enforcement|efp|mmpa|authority|accountability"),
    ("Science and Data",    r"knowledge|science|research|monitoring|uncertainty|engagement|uptake|assessment"),
    ("Land-Based Drivers",  r"agriculture|flooding|runoff|nutrient|sewage|pollution|spillway|river|watershed"),
]

# Summary-theme rules: collapse related concepts into a thematic super-node
_SUMMARY_RULES: List[Tuple[str, str]] = [
    ("Storms and Flooding",          r"storm|flood|hurricane|surge|rainfall|weather changes"),
    ("Dredging and Development",     r"dredging|dredge|development|construction|infrastructure|channelization|wakes|navigation"),
    ("Oil and Gas Industry",         r"oil|gas|rig|platform|lng|p&a|decommissioned|exploration|seismic"),
    ("Water Quality and Pollution",  r"water quality|pollution|nutrient|runoff|sewage|spillway|hypoxia|debris|dumping|trash|pesticides"),
    ("Fishery Management and Rules", r"management|regulation|quota|limit|enforcement|accountability|permit|reporting|compliance|hcr"),
    ("Discard and Release Mortality",r"discard|mortality|descending|barotrauma|release|culling|highgrading|post-release"),
    ("Seafood Markets and Trade",    r"price|demand|import|trade|ex-vessel|market|deficit"),
    ("Recreational Fishing Access",  r"access|angler|satisfaction|license"),
    ("Coastal and Marine Habitat",   r"habitat|nursery|estuar|marsh|grass|oyster|benthic|coral|reef fish habitat"),
    ("Marine Mammals and Predators", r"mammal|dolphin|manatee|shark|depredation|turtle|bird|pelican"),
    ("Reef Fish Species",            r"snapper|amberjack|trigger|reef fish|redfish|red drum|mackerel|grouper"),
    ("Bait and Forage",              r"bait|menhaden|poggy|anchovy|hard tail|forage|crabs|shrimp"),
    ("Climate and Physical Trends",  r"climate|sea-level|temp|salinity|current|trends|winter|subsidence|acidification|warming"),
    ("Social and Economic",          r"social|economic|well-being|community|jobs|cost|perception|h2b|aging out"),
]

_CATEGORY_COMPILED = [(label, re.compile(pat, re.IGNORECASE)) for label, pat in _CATEGORY_RULES]
_SUMMARY_COMPILED  = [(label, re.compile(pat, re.IGNORECASE)) for label, pat in _SUMMARY_RULES]


def get_category(name: str) -> str:
    if not name:
        return "Other"
    for label, rx in _CATEGORY_COMPILED:
        if rx.search(name):
            return label
    return "Other"


def get_summary_theme(name: str) -> str:
    """Return the summary theme for a node, or the node name itself if no rule matches."""
    if not name:
        return ""
    for label, rx in _SUMMARY_COMPILED:
        if rx.search(name):
            return label
    return name


def get_color(name: str) -> str:
    return FEP_COLORS[get_category(name)]


def build_node_lookup(nodes: List[str]) -> List[Dict[str, str]]:
    """Produce one lookup entry per node (mirrors the R `node_lookup` data-frame)."""
    return [
        {
            "OriginalNode": n,
            "Category":     get_category(n),
            "SummaryNode":  get_summary_theme(n),
            "HexColor":     get_color(n),
        }
        for n in nodes
    ]


## 8. Scenario simulation — `FCM/simulator.py`

Implements Kosko's propagation rule:

$$ a(t+1) = f\big(a(t) + A^{\!\top} a(t)\big) $$

where $A$ is the signed adjacency matrix, $a(t)$ is the activation vector at iteration $t$, and $f(\cdot)$ is a squashing function (sigmoid, tanh, or identity). Driver concepts are clamped to their initial value at every step so the resulting equilibrium answers the question *“what does the system look like if X stays elevated?”*.


> **Source — `FCM/simulator.py`**

In [ ]:
"""FCM scenario simulator — Kosko activation propagation with clamping.

Given a signed adjacency matrix A and an activation vector a(0), iterate:

        a(t+1) = f( a(t) + A^T · a(t) )

where f is a squashing function (sigmoid or tanh). Concepts marked as
"drivers" stay clamped to their initial value at every step (what-if
scenario analysis). Returns the full trajectory plus the equilibrium.
"""
from __future__ import annotations

from typing import Dict, List, Optional

import numpy as np


def _sigmoid(x: np.ndarray, lam: float = 1.0) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-lam * x))


class FCMSimulator:
    def __init__(self, concepts: List[str], adjacency: List[List[float]]):
        if not concepts:
            raise ValueError("concepts must be a non-empty list")
        self.concepts: List[str] = list(concepts)
        self.n: int = len(self.concepts)
        self.idx: Dict[str, int] = {c: i for i, c in enumerate(self.concepts)}
        self.A: np.ndarray = np.asarray(adjacency, dtype=float)
        if self.A.shape != (self.n, self.n):
            raise ValueError(
                f"adjacency shape {self.A.shape} does not match {self.n} concepts"
            )

    # ------------------------------------------------------------------
    def simulate(
        self,
        activation: Dict[str, float],
        steps: int = 30,
        squash: str = "sigmoid",
        lam: float = 1.0,
        clamp_drivers: bool = True,
        tol: float = 1e-4,
    ) -> Dict:
        """Propagate activation through the signed adjacency matrix.

        Args:
            activation: {concept_name: initial_value in [-1, 1]}.
            steps:      maximum iterations.
            squash:     "sigmoid" (default, outputs [0,1]) or "tanh" ([-1,1]).
            lam:        steepness of the sigmoid squashing.
            clamp_drivers: keep initially-set concepts at their initial value
                        every step (what-if analysis).
            tol:        convergence tolerance (max |Δa|).

        Returns: dict with trajectory, final state, convergence flag, and
            a per-concept "influence" (change from baseline).
        """
        if squash not in ("sigmoid", "tanh", "none"):
            raise ValueError(f"unknown squash={squash!r}")

        # Baseline initial state — user-provided concepts override, rest = 0
        a = np.zeros(self.n, dtype=float)
        driver_idx: List[int] = []
        for concept, value in activation.items():
            if concept in self.idx:
                i = self.idx[concept]
                a[i] = float(value)
                driver_idx.append(i)

        initial = a.copy()

        history: List[np.ndarray] = [a.copy()]
        converged = False

        for _ in range(steps):
            # Kosko propagation: each concept accumulates weighted inputs
            propagated = a + self.A.T @ a

            if squash == "sigmoid":
                new_a = _sigmoid(propagated, lam=lam)
                # Sigmoid outputs [0, 1]; rescale so baseline (0) maps to 0
                new_a = 2.0 * new_a - 1.0
            elif squash == "tanh":
                new_a = np.tanh(propagated * lam)
            else:
                new_a = propagated

            if clamp_drivers:
                for i in driver_idx:
                    new_a[i] = initial[i]

            diff = float(np.max(np.abs(new_a - a)))
            a = new_a
            history.append(a.copy())

            if diff < tol:
                converged = True
                break

        influence = a - initial
        ranked = sorted(
            ((c, float(a[i]), float(influence[i])) for i, c in enumerate(self.concepts)),
            key=lambda x: abs(x[2]),
            reverse=True,
        )

        return {
            "converged":   converged,
            "iterations":  len(history) - 1,
            "drivers":     list(activation.keys()),
            "squash":      squash,
            "lam":         lam,
            "final":       {c: float(a[i]) for i, c in enumerate(self.concepts)},
            "influence":   {c: float(influence[i]) for i, c in enumerate(self.concepts)},
            "top_effects": [
                {"concept": c, "final": final, "influence": inf}
                for c, final, inf in ranked[:15]
            ],
            "history": [
                {c: float(h[i]) for i, c in enumerate(self.concepts)}
                for h in history
            ],
        }


## 9. End-to-end demo (runnable)

All cells above are *source*. The cells below actually exercise the pipeline on a small synthetic text so reviewers can confirm what comes out of each stage. Run them top-to-bottom from the repo root with the project venv active.


In [ ]:
import sys, pathlib
# Make sure the repo root is on sys.path when running this notebook in place
ROOT = pathlib.Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'FCM').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print('Repo root:', ROOT)


In [ ]:
from FCM.extractor import CausalExtractor, normalize_concept, _is_clean_concept
from FCM.fcm_graph import FCMGraphBuilder
from FCM.simulator import FCMSimulator
from FCM.categorizer import get_category, get_summary_theme, get_color


### 9.1  Concept-name filters in action
Verify that the keyword filter rejects clause fragments, interrogatives, and useless modifiers while keeping legitimate domain phrases.


In [ ]:
samples = [
    'the very high fish stock',   # → Fish Stock
    'fish stock high',            # → Fish Stock (trailing modifier stripped)
    'How fish increase',          # → Fish Increase (leading interrogative stripped)
    'fish how grow',              # rejected (interrogative anywhere)
    'it is overfishing',          # → Overfishing
    'very high',                  # rejected (all modifiers)
    'fish fish stock',            # dedupe → Fish Stock
    'CPUE',                       # acronym preserved
    'water quality',              # legitimate
    'thing',                      # rejected (vague single)
    'climate change',             # legitimate
]
print(f"{'INPUT':<32} {'NORMALIZED':<28} CLEAN?")
print('-' * 72)
for s in samples:
    n = normalize_concept(s)
    ok = _is_clean_concept(n) if n else False
    print(f'{s!r:<32} {n!r:<28} {ok}')


### 9.2  Run the extractor on a small Gulf-FEI vignette


In [ ]:
vignette = (
    'Overfishing reduces fish biomass. '
    'Habitat loss threatens marine biodiversity. '
    'Climate change increases ocean temperature. '
    'Rising ocean temperature drives coral bleaching. '
    'Coral bleaching damages reef habitat. '
    'Strong fishery management restores fish biomass.'
)
extractor = CausalExtractor()
concepts, edges = extractor.extract([{'text': vignette, 'score': 1.0, 'doc_id': 'vignette'}])

print(f'Concepts ({len(concepts)}):')
for c in concepts:
    print(f'  - {c}  [{get_category(c)}]')
print()
print(f'Edges ({len(edges)}):')
for e in edges:
    sign = "+" if e.weight >= 0 else "-"
    print(f'  {e.source:<28} -[{sign}{abs(e.weight):.2f}]-> {e.target:<28} ({e.edge_type})')


### 9.3  Build the `FCMMap` and inspect its adjacency matrix


In [ ]:
import numpy as np
import pandas as pd
fcm_map = FCMGraphBuilder().build('demo query', concepts, edges)
matrix_df = pd.DataFrame(
    fcm_map.adjacency_matrix,
    index=fcm_map.concepts,
    columns=fcm_map.concepts,
).round(2)
matrix_df


### 9.4  Visualise the graph
We use the same publication renderer the web app calls.


In [ ]:
from src.FCM import FCMGenerator
from IPython.display import Image
gen = FCMGenerator(max_concepts=15, edge_threshold=0.15, directed=True)
G, details = gen.build_fcm(vignette, query='Demo: Gulf FEI vignette', num_results=20)
png_url = gen.plot_fcm(G, filename='walkthrough_demo', query='Demo: Gulf FEI vignette')
print('Plot saved at (app-relative):', png_url)
Image(filename=str(ROOT / png_url.lstrip('/')))


### 9.5  Run a what-if scenario
Clamp `Overfishing` high and `Fishery Management` high — see which concepts the system pushes up or down at equilibrium.


In [ ]:
sim = FCMSimulator(fcm_map.concepts, fcm_map.adjacency_matrix)
drivers = {}
for c in fcm_map.concepts:
    if 'Overfishing' in c:        drivers[c] = +1.0
    if 'Management' in c:         drivers[c] = +1.0
if not drivers:
    drivers[fcm_map.concepts[0]] = +1.0   # fallback so the cell still runs
result = sim.simulate(drivers, steps=40, squash='tanh', clamp_drivers=True)
print('Converged:', result['converged'], '| iterations:', result['iterations'])
print()
print('Top influenced concepts:')
for row in result['top_effects'][:8]:
    sign = '+' if row['influence'] >= 0 else '−'
    print(f"  {row['concept']:<28}  final={row['final']:+.3f}  Δ={sign}{abs(row['influence']):.3f}")


## 10. Quality controls and validation

Several guards keep the keyword set and graph honest:

| Guard | Where | What it does |
|---|---|---|
| Query meaningfulness | `main._is_meaningful_query` | Rejects empty / pure-stopword / interrogative-only queries before the RAG + FCM pipeline runs. |
| Concept normalisation | `extractor.normalize_concept` | Strips articles, leading/trailing useless modifiers, dedupes repeated tokens, preserves acronyms. |
| Concept cleanliness | `extractor._is_clean_concept` | Rejects clause fragments, interrogatives anywhere in the span, all-modifier phrases, sub-2-letter tokens, vague singles, and >50% stopword spans. |
| Polarity warnings | `main._build_response` | Flags maps that contain only positive (or only negative) edges so reviewers can sanity-check the source text. |
| Bidirectional dedup | `extractor._dedupe_bidirectional` | Drops same-sign reverse pairs (A→B and B→A both positive) keeping the stronger direction; preserves opposing-sign pairs as legitimate feedback loops. |
| Confidence-weighted merge | `extractor.fuzzy_merge_concepts` | Collapses near-duplicate concepts (plurals, fuzzy matches) into the longest canonical name and merges their edges by confidence-weighted mean. |
| 2:1 polarity interleave | `extractor._interleave_polarity` | Re-ranks final edges so positive and negative findings stay visible in the top-N. |

### Reproducing this notebook
Re-run `python -m FCM.docs._build_notebook` from the repo root after any change to the FCM modules. The script reads each module from disk and rebuilds the cells, so the notebook never drifts from the production code.
